In [19]:
import numpy as np

In [20]:
N = 10000
n = 15 #features
m = 2 #targets
D = [64,64] #number of hidden units
D = [n] + D + [m]
K = len(D)-2 #number of hidden layers


In [21]:
rng = np.random.default_rng(0)

In [22]:
h = 2*n
W1_true = rng.normal(-0.2,1.0,size=(h,n))
b1_true = rng.normal(-0.2,0.3, size=(h,1))
W2_true = rng.normal(-0.5,1.0,size=(m,h))
b2_true = rng.normal(-0.1,0.3, size=(m,1))

X = rng.uniform(-1.0,1.0,size=(n,N))

In [23]:
def ReLU(X):
    X = np.asarray(X)
    return np.maximum(0,X)

In [24]:
Y_clean = W2_true @ ReLU(W1_true @ X + b1_true) + b2_true # (m,N)
Y_clean.shape

(2, 10000)

In [25]:
noise_std = 0.1*np.std(Y_clean, axis=1,keepdims=True)
eps = rng.normal(0.0, noise_std, size=(m,N))
Y = Y_clean + eps
Y.shape

(2, 10000)

In [26]:
idx = rng.permutation(N)
val_frac = 0.2
n_val = int(val_frac*N)
val_idx = idx[:n_val]
train_idx = idx[n_val:]

In [27]:
X_train, Y_train = X[:,train_idx], Y[:,train_idx]
X_val, Y_val = X[:,val_idx], Y[:,val_idx]

In [28]:
X_train.shape, Y_train.shape, X_val.shape, Y_val.shape

((15, 8000), (2, 8000), (15, 2000), (2, 2000))

In [29]:
def init_params(n_in=n, D= D, n_out=m):
    Weights = []
    bias = []
    for l in range(1,len(D)):
        W = rng.normal(0.0, np.sqrt(2/D[l-1]),size=(D[l],D[l-1]))
        Weights.append(W)
        b = rng.normal(0.0, np.sqrt(2/D[l]),size=(D[l],1))
        bias.append(b)
    return Weights, bias 

In [30]:
def forward(X,W,b):
    H,Z = [X], [] # K+1,K | Z is shifted <- to H
    for l in range(len(W)):
        Z_l = W[l]@H[l]+b[l]
        if(l != len(W)-1):
            H_l = ReLU(Z_l)
        else:
            H_l = Z_l
        H.append(H_l); Z.append(Z_l)
    Y_hat = H[-1]
    return Y_hat, H,Z

In [31]:
def loss(Y_hat, Y):
    E = Y_hat - Y
    return 0.5 * np.mean(E**2)

In [32]:
def backprop(Y_hat,Y,H,Z,W):
    #Y_hat, Y: (m,n)
    #H[0]: (n,N) H[l]: (d_l, N)
    # Z[l]: (d_{l+1}, N)
    # W[l]: (d_{l+1}, d_l)

    N = Y.shape[1]
    E = Y_hat-Y
    L = len(W) #number of layers (K)

    ones = np.ones(shape=(N,1))


    dW, db = [None]*L, [None]*L
    G = E/N #(m,N)
    dW[L-1] = G @ H[L-1].T #(m,N)(N,d_K)
    db[L-1] = G @ ones #(m,N)(N,1)

    for l in range(L-2,-1,-1):
        dH = W[l+1].T @ G #(d+{l+1},N)
        
        mask = (Z[l]>0)#(d_{l+1},N)
        G = dH*mask

        dW[l] = G @ H[l].T #(d_{l+1},N)(N,d_l)
        db[l] = G @ ones #(d_{l+1},N)(N,1)
    
    return dW,db

In [33]:
def update_params(W, b, dW, db, lr):
    K = len(W)
    W_new = [W[l] - lr * dW[l] for l in range(K)]
    b_new = [b[l] - lr * db[l] for l in range(K)]
    return W_new, b_new

In [34]:
def train_step(X_batch, Y_batch, W,b, lr):
    Y_hat, H, Z = forward(X_batch,W,b)
    L = loss(Y_hat, Y_batch)
    dW,db = backprop(Y_hat,Y_batch,H,Z,W)
    W,b = update_params(W,b,dW,db,lr)
    return L, W, b

In [35]:
def fit(X_train, Y_train, W,b, lr=3e-3, epochs=100, batch_size=128,
        X_val=None, Y_val=None, rng=None, print_every=10):
    if rng is None:
        rng = np.random.default_rng(0)
    
    N = X_train.shape[1]
    history = {"train_loss":[], "val_loss":[]}

    for ep in range(1, epochs+1):
        idx = rng.permutation(N)
        train_losses = []

        for start in range(0,N, batch_size):
            batch_idx = idx[start:start+batch_size]
            
            Xb = X_train[:,batch_idx]
            Yb = Y_train[:,batch_idx]

            Lb,W,b= train_step(Xb,Yb,W,b,lr)
            train_losses.append(Lb)

        train_loss = float(np.mean(train_losses))
        history["train_loss"].append(train_loss)

        if X_val is not None and Y_val is not None:
            Y_hat_val,_,_ = forward(X_val,W,b)
            val_loss = float(loss(Y_hat_val,Y_val))
            history["val_loss"].append(val_loss)
            
        if (ep % print_every == 0) or (ep == 1) or (ep == epochs):
            if X_val is not None and Y_val is not None:
                print(f"epoch {ep:4d}/{epochs} | train_loss={train_loss:.6f} | val_loss={history['val_loss'][-1]:.6f}")
            else:
                print(f"epoch {ep:4d}/{epochs} | train_loss={train_loss:.6f}")
        
    return W,b, history

In [36]:
W, b = init_params(n,D,m)
W_new, b_new, history = fit(X_train,Y_train,W,b,X_val=X_val,Y_val=Y_val)
history

epoch    1/100 | train_loss=23.556877 | val_loss=10.189443
epoch   10/100 | train_loss=2.757106 | val_loss=2.781766
epoch   20/100 | train_loss=1.653776 | val_loss=1.729596
epoch   30/100 | train_loss=1.338880 | val_loss=1.458175
epoch   40/100 | train_loss=1.184632 | val_loss=1.299420
epoch   50/100 | train_loss=1.080041 | val_loss=1.207488
epoch   60/100 | train_loss=1.001076 | val_loss=1.136020
epoch   70/100 | train_loss=0.938265 | val_loss=1.080898
epoch   80/100 | train_loss=0.881860 | val_loss=1.020768
epoch   90/100 | train_loss=0.827639 | val_loss=0.968673
epoch  100/100 | train_loss=0.780957 | val_loss=0.935633


{'train_loss': [23.556876960086214,
  8.57105343819153,
  6.874479858829379,
  5.804196962784058,
  4.900977154942119,
  4.207620265818675,
  3.681437684016037,
  3.2990556839273495,
  2.999259580839063,
  2.7571063251296573,
  2.55291381730841,
  2.388139245611421,
  2.2397975020895924,
  2.1142715996561603,
  2.0077519480642407,
  1.9167825699989636,
  1.8384206485143006,
  1.768235931064405,
  1.7052544601232325,
  1.6537758550494988,
  1.6090701862947867,
  1.560303431122707,
  1.5297840977332773,
  1.4889089803272997,
  1.459454424054706,
  1.434196776435441,
  1.4038617088120018,
  1.384254638059623,
  1.3618369970655388,
  1.338880117841701,
  1.321893485956707,
  1.3027140257350052,
  1.2877380233493252,
  1.269483289768435,
  1.2555332972794542,
  1.2370296931862546,
  1.2256574754566651,
  1.2097023904061435,
  1.1983644025669868,
  1.1846317663041552,
  1.1708878971792185,
  1.1589060261832207,
  1.1504524959380282,
  1.137630938241202,
  1.1265288667770819,
  1.116742287501